In [75]:
with open('word_count.txt','r',encoding='utf-8') as f1:
    lines1=f1.readlines()
key_list1=[]
for item1 in lines1:
    key_list1.append(item1.split()[0])
key_list1[:4]

['ExtendedCognition', 'SituatedCognition', 'ExtendedMind', 'EmpiricalTest']

In [77]:
with open('keyword_cite_out_degree.txt','r',encoding='utf-8') as f2:
    lines2=f2.readlines()
key_list2=[]
for item2 in lines2:
    key_list2.append(item2.split()[0])
key_list2[:4]

['Multi-AgentSystem',
 'Agent-BasedModelingAndSimulation',
 'Intuition',
 'BoundedRationality']

In [78]:
with open('keyword_cite_in_degree.txt','r',encoding='utf-8') as f3:
    lines3=f3.readlines()
key_list3=[]
for item3 in lines3:
    key_list3.append(item3.split()[0])
key_list3[:4]

['Multi-AgentSystem',
 'Agent-BasedModelingAndSimulation',
 'Intuition',
 'BoundedRationality']

In [79]:
key_list = list(set(key_list1) & set(key_list2) & set(key_list3))
len(key_list)

379136

In [98]:
train_data={}
test_data={}
for item in lines1:
    if item.split()[0] in key_list[:1000]:
        train_list=[]
        test_list=[]
        for i in range(1,8):
            value1=[int(item.split()[i])]
            train_list.append(value1)
        for j in range(8,15):
            value2=[int(item.split()[j])]
            test_list.append(value2)
        train_data[item.split()[0]]=train_list
        test_data[item.split()[0]]=test_list

In [99]:
print(list(train_data.keys())[0])

U-Statistics


In [100]:
for item1 in lines2:
    if item1.split()[0] in key_list[:1000]:
        for i in range(1,8):
            train_data[item1.split()[0]][i-1].append(int(item1.split()[i]))
        for j in range(8,15):
            test_data[item1.split()[0]][j-12].append(int(item1.split()[j]))
print(train_data['U-Statistics'])
print(test_data['U-Statistics'])

[[4, 0], [0, 0], [5, 0], [2, 0], [0, 0], [3, 0], [2, 0]]
[[2, 11], [4, 0], [3, 0], [5, 0], [3, 0], [6, 0], [3, 10]]


In [101]:
for item2 in lines3:
    if item2.split()[0] in key_list[:1000]:
        for i in range(1,8):
            train_data[item2.split()[0]][i-1].append(int(item2.split()[i]))
        for j in range(8,15):
            test_data[item2.split()[0]][j-12].append(int(item2.split()[j]))
print(train_data['U-Statistics'])
print(test_data['U-Statistics'])

[[4, 0, 0], [0, 0, 0], [5, 0, 0], [2, 0, 0], [0, 0, 0], [3, 0, 0], [2, 0, 0]]
[[2, 11, 0], [4, 0, 0], [3, 0, 0], [5, 0, 0], [3, 0, 0], [6, 0, 0], [3, 10, 0]]


In [102]:
print(len(train_data))

1000


In [103]:
x=[]
y=[]
for key1 in train_data.keys():
    x.append(train_data[key1])
for key2 in test_data.keys():
    y.append(test_data[key2])
x[:4]

[[[4, 0, 0], [0, 0, 0], [5, 0, 0], [2, 0, 0], [0, 0, 0], [3, 0, 0], [2, 0, 0]],
 [[5, 0, 0],
  [3, 0, 0],
  [3, 0, 0],
  [5, 5, 17],
  [7, 0, 0],
  [2, 0, 0],
  [5, 0, 0]],
 [[2, 0, 0], [4, 0, 0], [0, 0, 0], [5, 0, 0], [2, 0, 0], [0, 0, 0], [3, 0, 4]],
 [[2, 0, 0], [0, 0, 0], [2, 0, 0], [0, 0, 0], [0, 0, 0], [2, 0, 7], [0, 0, 0]]]

In [104]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

In [110]:
x1=np.array(x)
y1=np.array(y)
x_train, x_test, y_train, y_test = train_test_split(x1, y1, test_size=0.2, random_state=42)
X_train_tensor = torch.tensor(x_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_test_tensor = torch.tensor(x_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# 定义 LSTM 模型（3层 LSTM）
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=3, dropout=0.2):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, 
                            batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        out, _ = self.lstm(x)
        last_output = out[:, -1, :]  # 取最后一个时间步的输出
        out = self.fc(last_output)
        return out

# 模型参数
input_dim = 3   # 输入特征数 (词频, 入度, 出度)
hidden_dim = 64 # 隐藏层神经元数
output_dim = 3  # 输出特征数 (预测词频, 入度, 出度)
num_layers = 3  # LSTM层数

# 初始化模型
model = LSTMModel(input_dim, hidden_dim, output_dim, num_layers)
criterion = nn.MSELoss()  # 损失函数
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  # 优化器

# 训练模型
num_epochs = 50
for epoch in range(num_epochs):
    model.train()
    for batch_X, batch_y in train_loader:
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

# 测试模型
model.eval()
with torch.no_grad():
    test_loss = 0
    for batch_X, batch_y in test_loader:
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        test_loss += loss.item()
    print(f"Test Loss: {test_loss / len(test_loader):.4f}")

# 测试样本预测
with torch.no_grad():
    sample_X = X_test_tensor[:5]  # 测试数据 (取前5个样本)
    predictions = model(sample_X)
    print("Predictions :", predictions.numpy())

RuntimeError: The size of tensor a (16) must match the size of tensor b (7) at non-singleton dimension 1